In [ ]:
import gc
import re
from peft import LoraConfig, get_peft_model
# ============================================================
#  FULL BENCHMARK BLOCK: 4-bit loading, loader, trainer, metrics
# ============================================================

import os
import sys
import time
import json

import torch
import pandas as pd
from torch.utils.data import Dataset

from tqdm.auto import tqdm

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel
from codebleu import calc_codebleu
from code_bert_score import score as codebert_score
import json

from datetime import datetime

RUN_TS = datetime.now().strftime("%Y%m%d_%H%M%S")
OUT_JSONL = f"benchmark_partial_{RUN_TS}.jsonl"

# ------------------------------------------------------------
# QUANTIZATION CONFIG 4-BIT 
# ------------------------------------------------------------


# 4-bit
bnb_config_4bit = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,  # su A100 va benissimo anche bfloat16
)

# 8-bit
bnb_config_8bit = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False,
)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=False,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    # o bfloat16 se la GPU lo supporta bene
)


import torch
from tqdm.auto import tqdm

@torch.inference_mode()
def generate_with_progress_stream(
    label,
    model,
    tok,
    prompts,
    max_src=512,
    max_new=512,
    bs=1,
):
    model.eval()

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    if getattr(model.config, "use_cache", None) is False:
        model.config.use_cache = True

    print(f"[{label}] Starting generation on {len(prompts)} prompts…")

    device = next(model.parameters()).device

    old_padding_side = tok.padding_side
    tok.padding_side = "right"

    n = len(prompts)

    for start in tqdm(range(0, n, bs), desc=f"[{label}] generating"):
        end = min(start + bs, n)
        batch_prompts = prompts[start:end]

        # Tokenizzazione SOLO del batch corrente, in CPU
        enc = tok(
            batch_prompts,
            padding=True,
            truncation=True,
            max_length=max_src,
            return_tensors="pt",
        )
        
        input_ids = enc["input_ids"]           # ancora su CPU
        attention_mask = enc["attention_mask"]
        
        # lunghezza vera di ogni prompt (numero di token non-pad)
        prompt_lens = attention_mask.sum(dim=1)        # (bs,)
        max_len = int(prompt_lens.max().item())        # max lunghezza nel batch
        
        # taglio in CPU
        input_ids = input_ids[:, :max_len]
        attention_mask = attention_mask[:, :max_len]
        
        # ora porto in GPU solo la parte utile
        input_ids = input_ids.to(device, non_blocking=True)
        attention_mask = attention_mask.to(device, non_blocking=True)


        gen_ids = model.generate(
            input_ids=input_ids,
            attention_mask=attention_mask,
            max_new_tokens=max_new,
            do_sample=False,
            num_beams=1,
            pad_token_id=tok.pad_token_id,
            eos_token_id=tok.eos_token_id,
            use_cache=True,
        )

        batch_gens = []
        for seq, plen in zip(gen_ids, prompt_lens.tolist()):
            plen = int(plen)
            gen_only = seq[plen:]
            text = tok.decode(gen_only, skip_special_tokens=True)
            batch_gens.append(text.strip())

        # facoltativo: rilascia tutto il possibile
        del enc, input_ids, attention_mask, prompt_lens, gen_ids, gen_only, seq


        # opzionale: ogni N batch puoi fare:
        # gc.collect()
        # torch.cuda.empty_cache()

        yield batch_gens


    tok.padding_side = old_padding_side

@torch.inference_mode()
def generate_one(
    model,
    tok,
    prompt: str,
    max_src: int = 512,
    max_new: int = 512,
):
    """
    Genera output per UN singolo prompt, greedy (no sampling).
    Ritorna una stringa.
    """
    model.eval()

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    if getattr(model.config, "use_cache", None) is False:
        model.config.use_cache = True

    device = next(model.parameters()).device

    # tokenizzazione singolo prompt (niente padding batch)
    enc = tok(
        prompt,
        truncation=True,
        max_length=max_src,
        return_tensors="pt",
    )
    input_ids = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    prompt_len = input_ids.size(1)

    gen_ids = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new,
        do_sample=False,          # greedy
        num_beams=1,
        pad_token_id=tok.pad_token_id,
        eos_token_id=tok.eos_token_id,
        use_cache=True,
    )

    gen_only = gen_ids[0, prompt_len:]
    text_raw = tok.decode(gen_only, skip_special_tokens=True)
    text = text_raw.strip()

    return text

@torch.inference_mode()
def generate_base_stream(
    label,
    model,
    tok,
    prompts,
    max_src=512,
    max_new=512,
):
    """
    Versione per i BASE: genera UNO prompt alla volta.
    Ritorna liste di 1 elemento (stessa interfaccia del batch).
    """
    model.eval()

    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id
    if getattr(model.config, "use_cache", None) is False:
        model.config.use_cache = True

    print(f"[{label}] Starting generation on {len(prompts)} prompts (single-sample)…")

    for prompt in tqdm(prompts, desc=f"[{label}] generating"):
        text = generate_one(
            model=model,
            tok=tok,
            prompt=prompt,
            max_src=max_src,
            max_new=max_new,
        )
        # mantengo compatibilità: lista di stringhe
        yield [text]


# ------------------------------------------------------------
# ROOT + DATAFRAME JSONL LOADER
# ------------------------------------------------------------
def detect_project_root():
    # punto di partenza (adatta se serve)
    root = r"C:\Users\ldesa\PycharmProjects\TAP basic\notebooks\fine_tuning"
    root = os.path.abspath(root)

    while True:
        if os.path.exists(os.path.join(root, "src")):
            break
        new_root = os.path.dirname(root)
        if new_root == root:  # siamo arrivati alla root (C:\)
            raise RuntimeError("Non trovo la cartella 'src'.")
        root = new_root

    if root not in sys.path:
        sys.path.append(root)
    print(f"Project root detected: {root}")
    return root

def load_df_real():
    root = detect_project_root()
    real_path = os.path.join(root, "data", "dataset", "applets", "applets_real_new.jsonl")
    df = pd.read_json(real_path, lines=True, orient="records")
    print(f"Loaded df_real shape={df.shape}")
    return df

# ------------------------------------------------------------
# DATASET MINIMALE
# ------------------------------------------------------------
class PromptDataset(Dataset):
    def __init__(self, tokenizer, prompts, max_source_length=512):
        enc = tokenizer(
            prompts,
            padding=False,
            truncation=True,
            max_length=max_source_length,
        )
        self.input_ids = enc["input_ids"]
        self.attention_mask = enc["attention_mask"]
        self.prompt_lens = [len(x) for x in self.input_ids]

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, idx):
        return {
            "input_ids": self.input_ids[idx],
            "attention_mask": self.attention_mask[idx],
        }

# ------------------------------------------------------------
# LOADING MODELLI (4-BIT) + TRAINER
# ------------------------------------------------------------
def prepare_tokenizer(name):
    tok = AutoTokenizer.from_pretrained(name, trust_remote_code=True)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    tok.padding_side = "left"   # puoi anche mettere "right", ma non è obbligatorio
    return tok

def load_base(name, quant_mode="4bit"):
    print(f"[BASE] Loading {name} with quant={quant_mode}…")
    tok = prepare_tokenizer(name)
    t0 = time.time()

    if quant_mode == "4bit":
        model = AutoModelForCausalLM.from_pretrained(
            name,
            trust_remote_code=True,
            device_map="cuda:0",        # su A100 metti direttamente la GPU
            torch_dtype=torch.float16,
            quantization_config=bnb_config_4bit,
        )
    elif quant_mode == "8bit":
        model = AutoModelForCausalLM.from_pretrained(
            name,
            trust_remote_code=True,
            device_map="cuda:0",
            torch_dtype=torch.float16,
            quantization_config=bnb_config_8bit,
        )
    elif quant_mode == "16bit":
        # nessuna quantizzazione: modello "pieno" in bf16 (A100-friendly)
        model = AutoModelForCausalLM.from_pretrained(
            name,
            trust_remote_code=True,
            device_map="cuda:0",
            torch_dtype=torch.bfloat16,
        )
    else:
        raise ValueError(f"Unknown quant_mode={quant_mode}")

    if getattr(model.config, "pad_token_id", None) is None:
        model.config.pad_token_id = tok.pad_token_id

    model.eval()
    print(f"[BASE] Loaded in {time.time()-t0:.1f}s")
    return model, tok

def load_peft(base_name, peft_path):
    print(f"[PEFT] Loading base={base_name} (4bit) + adapter={peft_path}")
    base_model, tok = load_base(base_name, quant_mode="4bit")
    t0 = time.time()
    model = PeftModel.from_pretrained(base_model, peft_path)
    model.eval()
    print(f"[PEFT] Loaded in {time.time()-t0:.1f}s")
    return model, tok

def extract_js_block(text: str) -> str:
    """
    Estrae il primo blocco di codice JavaScript racchiuso in triple backtick.
    Se non trova nulla, restituisce stringa vuota.
    """
    pattern = r"```(?:javascript|js)?\s*(.*?)```"
    m = re.search(pattern, text, re.DOTALL | re.IGNORECASE)
    if not m:
        return ""
    code = m.group(1).strip()
    return code

# ------------------------------------------------------------
# METRICHE
# ------------------------------------------------------------
from tqdm.auto import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
from rouge_score import rouge_scorer

def compute_metrics(preds, refs, lang="javascript"):
    cb = []
    bleu = []
    meteor = []
    rouge1 = []
    rouge2 = []
    rougeL = []

    smoothie = SmoothingFunction().method1
    rouge = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)

    pbar = tqdm(range(len(preds)), desc="[metrics]")
    for i in pbar:
        p = preds[i]
        r = refs[i]

        # tokenizzazione semplice a whitespace (per BLEU/METEOR)
        p_tok = p.split()
        r_tok = r.split()

        # ----- CodeBLEU per esempio -----
        res = calc_codebleu([r], [p], lang)
        cb.append(res["codebleu"])

        # ----- BLEU -----
        bleu_i = sentence_bleu(
            [r_tok],          # lista di riferimenti tokenizzati
            p_tok,            # ipotesi tokenizzata
            smoothing_function=smoothie,
        )
        bleu.append(bleu_i)

        # ----- METEOR (QUI era il bug) -----
        # references: Iterable[Iterable[str]]
        # hypothesis: Iterable[str]
        meteor_i = meteor_score([r_tok], p_tok)
        meteor.append(meteor_i)

        # ----- ROUGE (lavora su stringhe intere) -----
        r_scores = rouge.score(r, p)
        rouge1.append(r_scores["rouge1"].fmeasure)
        rouge2.append(r_scores["rouge2"].fmeasure)
        rougeL.append(r_scores["rougeL"].fmeasure)

    # ----- CodeBERTScore F1 (lista) -----
    _, _, F1, _ = codebert_score(preds, refs, lang=lang)
    f1 = [float(x) for x in F1]

    return cb, f1, bleu, meteor, rouge1, rouge2, rougeL

def free_model(model, tok=None):
    try:
        if model is not None:
            # sposta su CPU se possibile, per lasciare libera la GPU
            try:
                model.to("cpu")
            except Exception:
                pass
        # cancella riferimenti Python
        del model
        if tok is not None:
            del tok
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    except Exception as e:
        print(f"[WARN] free_model failed: {e}")


def append_results_to_jsonl(new_rows, columns, path=OUT_JSONL):
    """
    new_rows: lista di liste (stesso formato di 'results')
    columns:  lista di nomi delle colonne
    """
    with open(path, "a", encoding="utf-8") as f:
        for row in new_rows:
            rec = dict(zip(columns, row))
            json.dump(rec, f, ensure_ascii=False)
            f.write("\n")

def print_partial_summary(results, columns):
    df = pd.DataFrame(results, columns=columns)
    summary = df.groupby(["model", "variant", "quant"])[["codebleu", "codebert_f1"]].mean()
    print("\n=== Partial summary ===")
    print(summary)
    print("=======================\n")

root = detect_project_root()

MODEL_REGISTRY_BASE = {
    "deepseek": "deepseek-ai/deepseek-coder-6.7b-instruct",
    "gemma":    "google/codegemma-7b-it",
    "qwen":     "Qwen/Qwen2.5-Coder-7B-Instruct",
    "llama2":   "codellama/CodeLlama-7b-Instruct-hf",
}

MODEL_REGISTRY_PEFT = {
    "deepseek": os.path.join(root,"models/fine_tuning/coder/COLAB_deepseek-coder/best_model"),
    "gemma":    os.path.join(root,"models/fine_tuning/coder/COLAB_codegemma/best_model"),
    "qwen":     os.path.join(root,"models/fine_tuning/coder/COLAB_qwen-coder/best_model"),
    "llama2":   os.path.join(root,"models/fine_tuning/coder/COLAB_codellama/best_model"),
}

PRINT_EVERY = 10  # stampa ogni 10 batch (cambia a piacere)

test_size=4
# ------------------------------------------------------------
# BENCHMARK FINALE: zs → base, ft → peft
# ------------------------------------------------------------
df = load_df_real()
df = df[:test_size]          # come prima
max_src = 1024
max_new = 512
bs = 4
lang = "javascript"

zs   = df["instruct_zs"].tolist()
ft   = df["instruct_ft"].tolist()
refs = df["filter_code"].tolist()

results = []

columns = ["model", "variant", "quant", "row_id", "codebleu", "codebert_f1","bleu", "meteor", "rouge1", "rouge2", "rougeL", "generated"]
results = []
global_step = 0

for key in tqdm(MODEL_REGISTRY_BASE.keys(), desc="MODELS"):
    base_name = MODEL_REGISTRY_BASE[key]

    # ---------- PEFT (sempre 4-bit) sui prompt ft ----------
    if key in MODEL_REGISTRY_PEFT:
        model, tok = load_peft(base_name, MODEL_REGISTRY_PEFT[key])

        try:
            batch_start = 0
            for batch_gens in generate_with_progress_stream(
                f"{key}_PEFT_4bit", model, tok, ft,
                max_src=max_src, max_new=max_new, bs=bs
            ):
                batch_size = len(batch_gens)
                batch_end = batch_start + batch_size

                refs_batch = refs[batch_start:batch_end]
                cb_batch, f1_batch, bleu_batch, meteor_batch, r1_batch, r2_batch, rL_batch = compute_metrics(batch_gens, refs_batch, lang)
                new_rows=[]
                for i, (c, f, b, m, r1, r2, rL, g) in enumerate(
                    zip(cb_batch, f1_batch, bleu_batch, meteor_batch, r1_batch, r2_batch, rL_batch, batch_gens)
                ):
                    row_id = batch_start + i
                    new_rows.append([
                        key,
                        'peft',
                        '4bit',
                        row_id,
                        c,      # codebleu
                        f,      # codebert f1
                        b,      # BLEU
                        m,      # METEOR
                        r1,     # ROUGE-1
                        r2,     # ROUGE-2
                        rL,     # ROUGE-L
                        g,      # generation
                    ])


                results.extend(new_rows)
                append_results_to_jsonl(new_rows, columns)

                global_step += 1
                if global_step % PRINT_EVERY == 0:
                    print(f"\n[step {global_step}] Last block: model={key}, variant=peft, quant=4bit")
                    print_partial_summary(results, columns)

                batch_start = batch_end

                del refs_batch, cb_batch, f1_batch, batch_gens, new_rows

        finally:
            free_model(model, tok)

    # ---------- BASE (4 / 8 / 16 bit) sui prompt zs ----------
    for quant_mode in ["4bit", "8bit", "16bit"]:
        model, tok = load_base(base_name, quant_mode=quant_mode)

        try:
            batch_start = 0
            for batch_gens in generate_base_stream(
                f"{key}_BASE_{quant_mode}", model, tok, zs,
                max_src=max_src, max_new=max_new,
            ):
                batch_size = len(batch_gens)
                batch_end = batch_start + batch_size

                refs_batch = refs[batch_start:batch_end]
                cb_batch, f1_batch, bleu_batch, meteor_batch, r1_batch, r2_batch, rL_batch = compute_metrics(batch_gens, refs_batch, lang)
                new_rows=[]
                for i, (c, f, b, m, r1, r2, rL, g) in enumerate(
                    zip(cb_batch, f1_batch, bleu_batch, meteor_batch, r1_batch, r2_batch, rL_batch, batch_gens)
                ):
                    row_id = batch_start + i
                    new_rows.append([
                        key,
                        'base',
                        quant_mode,
                        row_id,
                        c,      # codebleu
                        f,      # codebert f1
                        b,      # BLEU
                        m,      # METEOR
                        r1,     # ROUGE-1
                        r2,     # ROUGE-2
                        rL,     # ROUGE-L
                        g,      # generation
                    ])


                results.extend(new_rows)
                append_results_to_jsonl(new_rows, columns)

                global_step += 1
                if global_step % PRINT_EVERY == 0:
                    print(f"\n[step {global_step}] Last block: model={key}, variant=base, quant={quant_mode}")
                    print_partial_summary(results, columns)

                batch_start = batch_end

                del refs_batch, cb_batch, f1_batch, batch_gens, new_rows

        finally:
            free_model(model, tok)

    # QUI, alla fine del ciclo sul singolo modello `key`,
    # non dovrebbero restare riferimenti a model/tok.
    # Per sicurezza ulteriore:
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [ ]:
import gc
import torch



In [ ]:
import pandas as pd
result_df=pd.read_json(OUT_JSONL,orient="records",lines=True)

result_df.groupby(["variant","quant"]).mean(["codebleu","codebert_f1","bleu","meteor","rouge1","rouge2","rougeL"])